<!-- NB_DOC_INTRO_V1 -->
# Time Series — Introduction et premiers pas

> 📚 **Doc thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Time Series & Traitement du Signal)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Premiers pas en **time series** : decomposition (tendance, saisonnalite, residus), tests de stationnarite (ADF, KPSS), autocorrelation (ACF, PACF), lag features, train/test split TEMPOREL (anti-leakage). Tout execute sur des donnees synthetiques + Air Passengers (classique).

## Plan

1. Setup + dataset (Air Passengers + synth)
2. Visualisation (line + heatmap calendrier)
3. Decomposition (additive vs multiplicative)
4. Stationnarite (ADF, KPSS tests)
5. Differentiation (ordre 1, ordre saisonnier)
6. ACF / PACF
7. Lag features pour ML
8. Train/test split TEMPOREL (TimeSeriesSplit)
9. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Dataset — Air Passengers (classique TS + saisonnalite)

In [ ]:
# Air Passengers (synthese reproductible — meme structure que le dataset Box-Jenkins 1976)
dates = pd.date_range("1949-01-01", periods=144, freq="MS")
trend = np.linspace(100, 600, 144)
season = 80 * np.sin(2 * np.pi * np.arange(144) / 12)
noise = np.random.normal(0, 20, 144)
y = trend + season + noise + 0.5 * trend * np.sin(2 * np.pi * np.arange(144) / 12) / 100
ts = pd.Series(y, index=dates, name="passengers")

print(f"Shape : {ts.shape}, range : {ts.min():.0f} - {ts.max():.0f}")
print(ts.head())

## 2. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Time series
axes[0].plot(ts.index, ts.values)
axes[0].set(title="Air Passengers (synthese)", ylabel="Passengers")
axes[0].grid(True, alpha=0.3)

# Heatmap calendaire (annee × mois)
pivot = ts.to_frame("y").assign(year=lambda x: x.index.year, month=lambda x: x.index.month)
pivot = pivot.pivot(index="year", columns="month", values="y")
im = axes[1].imshow(pivot.values, cmap="YlOrRd", aspect="auto")
axes[1].set_xticks(range(12)); axes[1].set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
axes[1].set_yticks(range(len(pivot.index))); axes[1].set_yticklabels(pivot.index)
axes[1].set(title="Heatmap calendaire (annee × mois)", ylabel="Year", xlabel="Month")
plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

## 3. Decomposition additive vs multiplicative

- **Additive** : `y(t) = T(t) + S(t) + R(t)` (saisonnalite d'amplitude constante)
- **Multiplicative** : `y(t) = T(t) × S(t) × R(t)` (saisonnalite proportionnelle a T)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 10))

for col, model_type in enumerate(["additive", "multiplicative"]):
    res = seasonal_decompose(ts, model=model_type, period=12)
    res.observed.plot(ax=axes[0, col], title=f"{model_type.capitalize()} — Observed")
    res.trend.plot(ax=axes[1, col], title="Trend")
    res.seasonal.plot(ax=axes[2, col], title="Seasonal")
    res.resid.plot(ax=axes[3, col], title="Residuals")
    for r in range(4):
        axes[r, col].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Stationnarite — ADF et KPSS

In [ ]:
def stationarity_tests(s, name=""):
    s = s.dropna()
    adf_stat, adf_p, *_ = adfuller(s)
    kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
    print(f"\n=== {name} ===")
    print(f"  ADF  : stat={adf_stat:.3f}, p={adf_p:.4f}  → {'STATIONNAIRE' if adf_p < 0.05 else 'non-stationnaire'} (H0: non-stat)")
    print(f"  KPSS : stat={kpss_stat:.3f}, p={kpss_p:.4f}  → {'non-stationnaire' if kpss_p < 0.05 else 'STATIONNAIRE'} (H0: stat)")

stationarity_tests(ts, "Serie originale")

## 5. Differentiation

In [ ]:
# Ordre 1 : y_t - y_{t-1}
ts_diff1 = ts.diff().dropna()
stationarity_tests(ts_diff1, "Apres diff ordre 1")

# Diff saisonnier (12 pour mensuel annuel)
ts_diff12 = ts.diff(12).dropna()
stationarity_tests(ts_diff12, "Apres diff saisonnier (12)")

# Combined : diff(1).diff(12)
ts_diff_both = ts.diff().diff(12).dropna()
stationarity_tests(ts_diff_both, "Apres diff(1).diff(12)")

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
ts.plot(ax=axes[0, 0], title="Original (non stationnaire)")
ts_diff1.plot(ax=axes[0, 1], title="Diff ordre 1")
ts_diff12.plot(ax=axes[1, 0], title="Diff saisonnier (12)")
ts_diff_both.plot(ax=axes[1, 1], title="Diff(1).Diff(12)")
plt.tight_layout(); plt.show()

## 6. ACF / PACF

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(ts.dropna(), lags=40, ax=axes[0, 0])
axes[0, 0].set_title("ACF original")
plot_pacf(ts.dropna(), lags=40, ax=axes[0, 1])
axes[0, 1].set_title("PACF original")
plot_acf(ts_diff_both.dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title("ACF apres diff(1).diff(12)")
plot_pacf(ts_diff_both.dropna(), lags=40, ax=axes[1, 1])
axes[1, 1].set_title("PACF apres diff(1).diff(12)")
plt.tight_layout(); plt.show()

**Lecture ACF/PACF** :
- ACF qui decline lentement → non stationnaire
- ACF significatif a lag 12 → saisonnalite annuelle
- PACF significatif a lag p → suggere AR(p)
- ACF significatif a lag q → suggere MA(q)

## 7. Lag features pour ML

Approche **'TS-as-tabular'** : transformer la serie en features (lags, rolling, datetime features) puis utiliser un modele ML classique (RF, LightGBM).

In [ ]:
def make_features(series, lags=[1, 2, 3, 12], roll_windows=[3, 12]):
    df = pd.DataFrame({"y": series})
    df = df.assign(
        month=lambda x: x.index.month,
        year=lambda x: x.index.year,
        quarter=lambda x: x.index.quarter,
        dayofweek=lambda x: x.index.dayofweek,
        # Fourier features (capture saisonnalite cyclique)
        sin12=lambda x: np.sin(2 * np.pi * x.index.month / 12),
        cos12=lambda x: np.cos(2 * np.pi * x.index.month / 12),
    )
    for lag in lags:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in roll_windows:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"] = df["y"].shift(1).rolling(w).std()
    return df

features = make_features(ts).dropna()
print(features.head().round(2))
print(f"\nShape : {features.shape}, NaN total : {features.isna().sum().sum()}")

## 8. Train/test split TEMPOREL — TimeSeriesSplit

**Crucial** : ne JAMAIS faire `train_test_split(shuffle=True)` sur TS → leak du futur dans le passe.

In [ ]:
# Split classique : derniers 20% comme test
split_idx = int(len(features) * 0.8)
train, test = features.iloc[:split_idx], features.iloc[split_idx:]
print(f"Train : {len(train)} ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test  : {len(test)}  ({test.index.min().date()} → {test.index.max().date()})")

# TimeSeriesSplit (pour cross-validation)
tscv = TimeSeriesSplit(n_splits=5)
fig, ax = plt.subplots(figsize=(10, 4))
for i, (tr_idx, te_idx) in enumerate(tscv.split(features)):
    ax.plot(features.index[tr_idx], np.ones(len(tr_idx)) * i, "b.", alpha=0.5)
    ax.plot(features.index[te_idx], np.ones(len(te_idx)) * i, "r.", alpha=0.7)
ax.set_yticks(range(5)); ax.set_yticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_title("TimeSeriesSplit (bleu = train, rouge = test)")
plt.tight_layout(); plt.show()

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `train_test_split(shuffle=True)` sur TS | TimeSeriesSplit ou split manuel par date |
| Fillna(0) sur TS | `interpolate(method='time')` ou `ffill` |
| Pas tester stationnarite | ADF + KPSS — guide pour ARIMA |
| Conclure 'pas saisonnier' sans ACF | Toujours ACF / PACF |
| Lag features sans `shift(1)` dans rolling | Leak (rolling inclut t actuel) |
| Pas de Fourier features pour saisonnalite | Ajouter sin/cos pour periode connue |
| Imputation par mean global | Mauvaise idee — utiliser interpolation locale |
| Cross-validation random | TimeSeriesSplit ou walk-forward |


## References

### Documentation
- statsmodels TSA : https://www.statsmodels.org/stable/tsa.html
- *Forecasting: Principles and Practice* (Hyndman) — gratuit : https://otexts.com/fpp3/
- Brownlee datasets : https://github.com/jbrownlee/Datasets

### Voir aussi
- [TS_Time_Series_Overview.ipynb](TS_Time_Series_Overview.ipynb)
- [TS_ARIMAs_Revoir.ipynb](TS_ARIMAs_Revoir.ipynb)
- [TS_Generer_Sequence.ipynb](TS_Generer_Sequence.ipynb)
- [TS_Maintenance_Predictive_GOOD.ipynb](TS_Maintenance_Predictive_GOOD.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
